## Predicting Trip Duration (Routing AI)
**Objective:** Predicting the exact duration (in minutes) of a taxi trip based on pre-trip logistics (distance, time of day, and location) to improve ETA accuracy and fleet routing efficiency.
- Distance alone does not equal time in NYC. A 2-mile trip at 3:00 AM takes 5 minutes, but at 5:00 PM it can take 35 minutes.
- Inaccurate routing estimates frustrate passengers and prevent fleet managers from accurately forecasting when a vehicle will be free for its next dispatch.
- Automating duration estimation allows algorithms to learn the hidden, non-linear traffic patterns of the city.

## Hypotheses:
1. The Physics Over Human Behavior: Unlike predicting tips (which relies on subjective human generosity), predicting duration is rooted in physics and logistics.Therefore, a predictive model should achieve a significantly higher accuracy score ($R^2 > 0.75$) than behavioral models.
2. Temporal Traffic Penalties: The time of day (pickup_hour) will act as a massive multiplier on trip duration, specifically punishing trips that occur during the Evening Rush (16:00 - 19:00).
3. Non-Linear Model Necessity: Because traffic congestion scales exponentially rather than linearly, an algorithm like XGBoost will be required to capture the sudden spikes in travel time.

In [19]:
import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
set_config(display="text")

In [15]:
# 1. Load the pristine data
con = duckdb.connect()
clean_data_path = 'data/cleaned_taxi_data_for_ml.parquet'
df = con.execute(f"SELECT * FROM '{clean_data_path}'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
# 2. FEATURE ENGINEERING: Calculate trip duration in exact minutes
print("⏱️ Calculating actual trip durations...")
df['duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

⏱️ Calculating actual trip durations...


In [7]:
# 3. CLEANING: Remove impossible durations (under 1 minute or over 2 hours)
df_filtered = df[(df['duration_minutes'] >= 1) & (df['duration_minutes'] <= 120)].copy()

In [9]:
# 4. SELECT FEATURES: What determines traffic? Distance, Time, and Location.
features = ['pickup_hour', 'day_of_week', 'trip_distance', 'pickup_borough', 'dropoff_borough']
X_raw = df_filtered[features]
y = df_filtered['duration_minutes']

In [11]:
# 5. ONE-HOT ENCODING & DOWNSAMPLING
print("🧮 Encoding locations and prepping data...")
X = pd.get_dummies(X_raw, columns=['pickup_borough', 'dropoff_borough'], drop_first=True)

🧮 Encoding locations and prepping data...


In [13]:
# Sample 150,000 trips to keep training fast
X_sample = X.sample(n=150000, random_state=42)
y_sample = y.loc[X_sample.index]
X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

In [21]:
# 6. TRAIN XGBOOST
print("🧠 Training the XGBoost Traffic AI... (Please wait)")
xgb_model = XGBRegressor(n_estimators=150, max_depth=7, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

🧠 Training the XGBoost Traffic AI... (Please wait)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=150, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [23]:
# 7. GRADE THE EXAM
preds = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

In [25]:
print("\n🎯 --- ROUTING AI REPORT CARD --- 🎯")
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"R-Squared (Accuracy Score): {r2:.3f}")

# Let's look at a few real predictions vs reality!
results_df = pd.DataFrame({
    'Actual Duration (Mins)': y_test.values[:5].round(1),
    'AI Predicted Duration (Mins)': preds[:5].round(1),
    'Error (Mins)': abs(y_test.values[:5] - preds[:5]).round(1)
})
print("\n🔍 First 5 Predictions vs Reality:")
display(results_df)


🎯 --- ROUTING AI REPORT CARD --- 🎯
Mean Absolute Error (MAE): 3.94 minutes
R-Squared (Accuracy Score): 0.813

🔍 First 5 Predictions vs Reality:


,Actual Duration (Mins),AI Predicted Duration (Mins),Error (Mins)
0,18.1,11.000000,7.1
1,2.8,5.500000,2.7
2,17.0,20.799999,3.8
3,9.4,11.500000,2.1
4,18.9,19.299999,0.4


In [28]:
import joblib
joblib.dump(xgb_model, 'duration_model.pkl')
joblib.dump(list(X_train.columns), 'duration_columns.pkl')
print("✅ Duration Model Saved!")

✅ Duration Model Saved!
